[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 困难：专家混合（MoE）

实现一个 **专家混合** 层（Mixtral / Switch Transformer 风格）。

### 函数签名
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### 架构
- `self.router`：`nn.Linear(d_model, num_experts)` — 门控网络
- `self.experts`：由 MLP 组成的 `nn.ModuleList` `(Linear→ReLU→Linear)`
- 对每个 token：选择 top-k 个专家，计算其输出的加权和

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn 

In [ ]:
# ✏️ 在此实现你的代码

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        pass  # router + experts

    def forward(self, x):
        pass  # 将 tokens 路由到 top-k 专家

In [ ]:
import torch.nn.functional as F
from typing import Tuple

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int = 2):
        """
        初始化专家混合层
        
        Args:
            d_model: 输入/输出维度
            d_ff: 前馈网络的隐藏层维度
            num_experts: 专家网络的数量
            top_k: 每个token选择的专家数量
        """
        super().__init__()
        
        self.d_model = d_model
        self.d_ff = d_ff
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 路由网络：为每个token计算专家选择分数
        self.router = nn.Linear(d_model, num_experts, bias=False)
        
        # 专家网络：每个专家是一个两层的MLP (Linear -> ReLU -> Linear)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model)
            )
            for _ in range(num_experts)
        ])
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播
        
        Args:
            x: 输入张量，形状为 (batch_size, seq_len, d_model)
            
        Returns:
            输出张量，形状为 (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, d_model = x.shape
        assert d_model == self.d_model, f"期望d_model={self.d_model}，但得到{d_model}"
        
        # 重塑为 (batch_size * seq_len, d_model) 以便对每个token独立处理
        x_flat = x.reshape(-1, d_model)  # (B*S, D)
        num_tokens = x_flat.shape[0]
        
        # 1. 计算路由分数
        router_logits = self.router(x_flat)  # (B*S, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)  # (B*S, num_experts)
        
        # 2. 选择top-k个专家
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # top_k_probs: (B*S, top_k)
        # top_k_indices: (B*S, top_k)
        
        # 归一化top-k的权重
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        # 3. 计算每个专家的输出
        # 初始化输出张量
        output = torch.zeros_like(x_flat)  # (B*S, D)
        
        # 对每个专家单独处理，将每个 token 都找到 topk 个专家的概率分布，然后将特征维度对专家进行 MLP，再乘上专家的概率
        for expert_idx, expert in enumerate(self.experts):
            # 找出哪些 token选择了这个专家
            # top_k_indices 中哪些位置等于 expert_idx
            mask = (top_k_indices == expert_idx)  # (B*S, top_k)
            
            if mask.any():
                # 获取选择了当前专家的token索引
                token_indices = mask.any(dim=-1).nonzero(as_tuple=True)[0]  # (num_selected,)
                
                if len(token_indices) > 0:
                    # 获取这些token的输入
                    selected_x = x_flat[token_indices]  # (num_selected, D)
                    
                    # 计算专家的输出
                    expert_output = expert(selected_x)  # (num_selected, D)
                    
                    # 获取对应的权重
                    # 对于每个选中的token，找到它对应的专家权重
                    selected_weights = torch.zeros(len(token_indices), dtype=torch.float32, device=x.device)
                    
                    for i, token_idx in enumerate(token_indices):
                        # 找到这个token在top_k_indices中对应expert_idx的位置
                        token_top_k = top_k_indices[token_idx]  # (top_k,)
                        weight_mask = (token_top_k == expert_idx)  # (top_k,)
                        if weight_mask.any():
                            # 获取对应的权重
                            selected_weights[i] = top_k_probs[token_idx][weight_mask].item()
                    
                    # 将加权输出添加到总输出中
                    output[token_indices] += selected_weights.unsqueeze(-1) * expert_output
        
        # 重塑回原始形状
        output = output.reshape(batch_size, seq_len, d_model)
        
        return output

    def get_router_weights(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        获取路由权重和选中的专家索引（用于分析和调试）
        
        Args:
            x: 输入张量，形状为 (batch_size, seq_len, d_model)
            
        Returns:
            top_k_probs: (batch_size, seq_len, top_k) 选中的专家权重
            top_k_indices: (batch_size, seq_len, top_k) 选中的专家索引
        """
        batch_size, seq_len, d_model = x.shape
        x_flat = x.reshape(-1, d_model)
        
        router_logits = self.router(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        return top_k_probs.reshape(batch_size, seq_len, self.top_k), \
               top_k_indices.reshape(batch_size, seq_len, self.top_k)


# 测试代码
if __name__ == "__main__":
    # 设置参数
    d_model = 512
    d_ff = 2048
    num_experts = 8
    top_k = 2
    batch_size = 4
    seq_len = 10
    
    # 创建模型
    model = MixtureOfExperts(d_model, d_ff, num_experts, top_k)
    
    # 创建随机输入
    x = torch.randn(batch_size, seq_len, d_model)
    
    # 前向传播
    output = model(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    
    # 检查输出形状是否正确
    assert output.shape == x.shape, "输出形状应与输入相同"
    
    # 测试路由权重
    weights, indices = model.get_router_weights(x)
    print(f"路由权重形状: {weights.shape}")
    print(f"专家索引形状: {indices.shape}")
    
    print("\n✓ 所有测试通过!")

In [ ]:
# 🧪 调试
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('输出:', moe(x).shape)
print('参数数:', sum(p.numel() for p in moe.parameters()))

In [ ]:
# ✅ 提交
from torch_judge import check
check('moe')